# 1.2 — Preprocessing and Gait Segmentation

This notebook validates the preprocessing pipeline end-to-end on **synthetic Vicon data**.

The processing logic lives in `shared/gait_processing/`. This notebook covers:

- loading configuration and one trial,
- inspecting input quality,
- cleaning kinematic and analog signals,
- detecting gait events,
- validating detected events against synthetic ground truth,
- building gait cycles,
- normalizing cycles to 0–100% gait cycle,
- visualizing intermediate and final results.

| | |
|---|---|
| **Inputs** | `data/raw/{subject}/{trial}/` (trajectories, analog, events, artifacts CSVs) |
| **Outputs** | Normalized gait-cycle table, pipeline summary |

> **Important:** the current dataset is synthetic. The goal is to stabilize and validate the pipeline architecture before applying the same workflow to real Vicon data.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "shared") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "shared"))

DATA_ROOT = PROJECT_ROOT / "1-experimentation" / "data" / "raw"
CONFIG_PATH = PROJECT_ROOT / "config" / "processing.yaml"

print("Project root:", PROJECT_ROOT)
print("Data root:", DATA_ROOT)
print("Config:", CONFIG_PATH)

In [ ]:
from gait_processing import ProcessingConfig, load_trial
from gait_processing.quality import check_trial_quality
from gait_processing.cleaning import (
    clean_kinematic_trial,
    clean_analog_trial,
)
from gait_processing.events import (
    contact_to_events,
    detect_gait_events,
    event_timing_error,
)
from gait_processing.segmentation import (
    build_gait_cycles,
    time_normalize_cycles,
)

## 2. Load processing configuration

In [ ]:
cfg = ProcessingConfig.from_yaml(CONFIG_PATH)
cfg

In [ ]:
print("Kinematic sampling rate:", cfg.kinematic_fs_hz, "Hz")
print("Analog sampling rate:", cfg.analog_fs_hz, "Hz")
print("Marker cutoff:", cfg.marker_cutoff_hz, "Hz")
print("GRF cutoff:", cfg.grf_cutoff_hz, "Hz")
print("Max short kinematic gap:", cfg.max_short_gap_s, "s")
print("Max short analog gap:", cfg.analog_max_short_gap_s, "s")
print("Normalized points:", cfg.normalized_points)

## 3. Load one trial

For development we use one known synthetic acquisition. Later, the same notebook can be pointed to a real trial without changing the downstream processing code.

In [ ]:
SUBJECT_ID = "Sub01"
TRIAL_ID = "trial0001"

trial = load_trial(
    root=DATA_ROOT,
    subject_id=SUBJECT_ID,
    trial_id=TRIAL_ID,
)

trial

In [ ]:
print(
    trial.subject_id,
    trial.trial_id,
    trial.kinematic_fs_hz,
    trial.analog_fs_hz,
    trial.duration_s,
    trial.mass_kg,
    trial.height_m,
    trial.cohort,
    trial.condition,
)

In [ ]:
display(trial.trajectories.head())
display(trial.analog.head())
display(trial.subject.to_frame("value"))
display(trial.trial.to_frame("value"))

## 4. Raw-data quality control

In [ ]:
quality = check_trial_quality(trial)

print("Kinematic integrity")
display(quality.kinematic_integrity)

print("Analog integrity")
display(quality.analog_integrity)

In [ ]:
print("Kinematic channels with missing data:")
display(
    quality.kinematic_channels[
        quality.kinematic_channels["missing_n"] > 0
    ].sort_values("missing_n", ascending=False)
)

print("Analog channels with missing data:")
display(
    quality.analog_channels[
        quality.analog_channels["missing_n"] > 0
    ].sort_values("missing_n", ascending=False)
)

### QC interpretation

At this stage the signals are **not modified**. QC only characterizes the input and checks structural/temporal integrity.

## 5. Cleaning

In [ ]:
clean_kin, kin_report = clean_kinematic_trial(
    trial.trajectories,
    cfg,
)

clean_analog, analog_report = clean_analog_trial(
    trial.analog,
    cfg,
)

print("Kinematic cleaning report")
display(kin_report)

print("Analog cleaning report")
display(analog_report)

In [ ]:
print("Kinematic channels where values were interpolated:")
display(
    kin_report.channel_report[
        kin_report.channel_report["interpolated_values"] > 0
    ].sort_values("interpolated_values", ascending=False)
)

print("Kinematic channels with unresolved missing values:")
display(
    kin_report.channel_report[
        kin_report.channel_report["final_missing"] > 0
    ].sort_values("final_missing", ascending=False)
)

### Plot 1 — Raw vs cleaned marker with a long gap

`RKNE_Y_mm` is useful for validating the conservative missing-data policy: long gaps must remain missing rather than being fabricated by interpolation.

In [ ]:
MARKER_CHANNEL = "RKNE_Y_mm"

fig, ax = plt.subplots(figsize=(11, 4))

ax.plot(
    trial.trajectories["time_s"],
    trial.trajectories[MARKER_CHANNEL],
    label="Raw",
    alpha=0.65,
)

ax.plot(
    clean_kin["time_s"],
    clean_kin[MARKER_CHANNEL],
    label="Cleaned",
    linewidth=2,
)

ax.set_xlabel("Time [s]")
ax.set_ylabel("Marker position [mm]")
ax.set_title(f"Raw vs cleaned — {MARKER_CHANNEL}")
ax.legend()
ax.grid(alpha=0.2)

plt.show()

### Plot 2 — Raw vs cleaned vertical GRF

In [ ]:
GRF_CHANNEL = "L_GRF_V_N"

fig, ax = plt.subplots(figsize=(11, 4))

ax.plot(
    trial.analog["time_s"],
    trial.analog[GRF_CHANNEL],
    label="Raw",
    alpha=0.55,
)

ax.plot(
    clean_analog["time_s"],
    clean_analog[GRF_CHANNEL],
    label="Cleaned",
    linewidth=2,
)

ax.set_xlabel("Time [s]")
ax.set_ylabel("Vertical GRF [N]")
ax.set_title(f"Raw vs cleaned — {GRF_CHANNEL}")
ax.legend()
ax.grid(alpha=0.2)

plt.show()

## 6. Gait-event detection

In [ ]:
events, contacts, event_report = detect_gait_events(
    clean_analog,
    mass_kg=trial.mass_kg,
    cfg=cfg,
)

event_report

In [ ]:
display(events.head(20))

## 7. Synthetic ground-truth validation

The synthetic dataset contains `L_contact_true` and `R_contact_true`. We use these labels only for validation of the detector; the detector itself uses vertical GRF.

In [ ]:
truth_frames = []

for side in ("L", "R"):
    truth_side = contact_to_events(
        time_s=trial.analog["time_s"].to_numpy(),
        sample=trial.analog["sample"].to_numpy(),
        contact=trial.analog[f"{side}_contact_true"].to_numpy(),
        side=side,
        source="synthetic_ground_truth",
    )
    truth_frames.append(truth_side)

truth_events = (
    pd.concat(truth_frames, ignore_index=True)
    .sort_values(["time_s", "side"])
    .reset_index(drop=True)
)

display(truth_events.head(20))

In [ ]:
timing_errors = event_timing_error(
    detected=events,
    truth=truth_events,
)

timing_summary = (
    timing_errors
    .assign(
        abs_error_ms=lambda x: x["error_ms"].abs()
    )
    .groupby(["side", "event"])
    .agg(
        n=("error_ms", "count"),
        bias_ms=("error_ms", "mean"),
        mae_ms=("abs_error_ms", "mean"),
        std_ms=("error_ms", "std"),
        max_abs_error_ms=("abs_error_ms", "max"),
    )
)

timing_summary

### Plot 3 — Vertical GRF with detected and ground-truth gait events

Detected events are shown together with the synthetic reference. Small systematic offsets are expected from the hysteresis thresholds.

In [ ]:
SIDE = "L"
grf_col = f"{SIDE}_GRF_V_N"

fig, ax = plt.subplots(figsize=(13, 4))

ax.plot(
    clean_analog["time_s"],
    clean_analog[grf_col],
    label=f"{SIDE} vertical GRF",
    linewidth=1.5,
)

detected_side = events[events["side"] == SIDE]
truth_side = truth_events[truth_events["side"] == SIDE]

for _, row in detected_side.iterrows():
    ax.axvline(
        row["time_s"],
        linestyle="--" if row["event"] == "Heel Strike" else ":",
        alpha=0.65,
    )

for _, row in truth_side.iterrows():
    ax.axvline(
        row["time_s"],
        linestyle="-.",
        alpha=0.35,
    )

ax.set_xlabel("Time [s]")
ax.set_ylabel("Vertical GRF [N]")
ax.set_title(
    f"{SIDE} GRF with detected HS/TO and synthetic ground truth"
)
ax.grid(alpha=0.2)

plt.show()

## 8. Gait-cycle segmentation

In [ ]:
cycles, segmentation_report = build_gait_cycles(
    events,
    cfg,
)

segmentation_report

In [ ]:
display(
    cycles[
        [
            "cycle_id",
            "side",
            "hs_start_s",
            "hs_end_s",
            "duration_s",
            "valid",
            "rejection_reason",
        ]
    ]
)

### Plot 4 — Gait-cycle durations

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

valid_cycles = cycles[cycles["valid"]].copy()

for side, group in valid_cycles.groupby("side"):
    ax.scatter(
        group["cycle_id"],
        group["duration_s"],
        label=side,
    )

ax.axhline(
    cfg.min_cycle_s,
    linestyle="--",
    label="Minimum allowed",
)

ax.axhline(
    cfg.max_cycle_s,
    linestyle=":",
    label="Maximum allowed",
)

ax.set_xlabel("Cycle")
ax.set_ylabel("Duration [s]")
ax.set_title("Accepted gait-cycle durations")
ax.tick_params(axis="x", rotation=60)
ax.legend()
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

## 9. Time normalization to 0–100% gait cycle

In [ ]:
normalized_cycles = time_normalize_cycles(
    clean_kin,
    cycles,
    cfg,
)

print("Normalized shape:", normalized_cycles.shape)

print("\nSamples per cycle:")
display(
    normalized_cycles
    .groupby("cycle_id")
    .size()
    .value_counts()
)

print("\nVariables with remaining NaNs:")
display(
    normalized_cycles
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

### Plot 5 — Overlay of normalized knee-flexion cycles

This plot is more biomechanically informative than global marker coordinates because joint-angle trajectories should show a repeatable gait-cycle pattern.

In [ ]:
ANGLE_CHANNEL = "L_KneeFlex_deg"

fig, ax = plt.subplots(figsize=(10, 5))

left_cycles = normalized_cycles[
    normalized_cycles["side"] == "L"
]

for cycle_id, cycle_data in left_cycles.groupby("cycle_id"):
    ax.plot(
        cycle_data["gait_pct"],
        cycle_data[ANGLE_CHANNEL],
        alpha=0.45,
    )

ax.set_xlabel("Gait cycle [%]")
ax.set_ylabel("Angle [deg]")
ax.set_title(f"Normalized left gait cycles — {ANGLE_CHANNEL}")
ax.grid(alpha=0.2)

plt.show()

### Plot 6 — Mean +/- standard deviation of normalized cycles

This is still a descriptive visualization. Construction of normative references belongs to a later stage.

In [ ]:
angle_summary = (
    left_cycles
    .groupby("gait_pct")[ANGLE_CHANNEL]
    .agg(["mean", "std"])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    angle_summary["gait_pct"],
    angle_summary["mean"],
    label="Mean",
)

ax.fill_between(
    angle_summary["gait_pct"],
    angle_summary["mean"] - angle_summary["std"],
    angle_summary["mean"] + angle_summary["std"],
    alpha=0.2,
    label="+/-1 SD",
)

ax.set_xlabel("Gait cycle [%]")
ax.set_ylabel("Angle [deg]")
ax.set_title(f"Mean normalized trajectory — {ANGLE_CHANNEL}")
ax.legend()
ax.grid(alpha=0.2)

plt.show()

## 10. Pipeline summary

In [ ]:
summary = pd.DataFrame(
    {
        "metric": [
            "Subject",
            "Trial",
            "Kinematic sampling [Hz]",
            "Analog sampling [Hz]",
            "Original kinematic missing values",
            "Interpolated kinematic values",
            "Final kinematic missing values",
            "Left contact accuracy",
            "Right contact accuracy",
            "Candidate cycles",
            "Accepted cycles",
            "Rejected cycles",
            "Normalized points per cycle",
        ],
        "value": [
            trial.subject_id,
            trial.trial_id,
            trial.kinematic_fs_hz,
            trial.analog_fs_hz,
            kin_report.original_missing,
            kin_report.interpolated_values,
            kin_report.final_missing,
            event_report.contact_accuracy_left,
            event_report.contact_accuracy_right,
            segmentation_report.candidate_cycles,
            segmentation_report.accepted_cycles,
            (
                segmentation_report.candidate_cycles
                - segmentation_report.accepted_cycles
            ),
            cfg.normalized_points,
        ],
    }
)

summary

## 11. Interpretation / next step

At this point the preprocessing pipeline has been validated on one synthetic trial:

1. raw data can be loaded with subject/trial metadata,
2. QC checks timing and missingness without modifying data,
3. short gaps are interpolated conservatively while long gaps remain missing,
4. GRF-based gait events are detected and validated against synthetic truth,
5. same-side HS-to-HS cycles are constructed,
6. accepted cycles are normalized to 101 points from 0-100%.

The next step is `1_3_data_preprocessed_eda.ipynb` for exploratory analysis, followed by feature engineering in `1_4`.